# Create long-format versions of timeseries case data

This script reads the snakebite data from the google drive, extracts the category specific (poison/non-poison) case data and converts it to long form with date conversion.

In [35]:
# import argparse
import re
import sys
import os
from pathlib import Path

import pandas as pd
import datetime
import nepali_datetime

In [36]:
df_pois = pd.read_csv(
    '/n/holylabs/LABS/cgolden_lab/Lab/data_freeze/golden_googledrive_rclone/Climate-Smart Public Health - Nepal/4. Datasets/snake_bites/Poisnous snake bite cases.csv',
    dtype={"organisationunitcode": str}).fillna(0)

df_nonpois = pd.read_csv(
    '/n/holylabs/LABS/cgolden_lab/Lab/data_freeze/golden_googledrive_rclone/Climate-Smart Public Health - Nepal/4. Datasets/snake_bites/Non-poisonous snake bite cases.csv',
    dtype={"organisationunitcode": str}).fillna(0)

In [37]:
keep_cols_p = ['organisationunitid',
 'organisationunitname',
 'organisationunitcode'] + [col for col in df_pois.columns if ('OPD' in col) and len(col) < 85]

keep_cols_np = ['organisationunitid',
 'organisationunitname',
 'organisationunitcode'] + [col for col in df_nonpois.columns if ('OPD' in col) and len(col) < 85]

In [38]:
NEPALI_MONTHS = ['Baishak',
    'Jestha',
    'Asar',
    'Shrawan',
    'Bhadra',
    'Ashwin',
    'Kartik',
    'Mangsir',
    'Poush',
    'Magh',
    'Falgun',
    'Chaitra']

MONTH_PATTERN = re.compile(
    r"(" + "|".join(re.escape(m) for m in NEPALI_MONTHS) + r")\s+(\d{4})",
    re.IGNORECASE,
)

In [39]:
def extract_period(col_name: str) -> str | None:
    """Return 'Month YYYY' if found in col_name, else None."""
    match = MONTH_PATTERN.search(col_name)
    if match:
        month = match.group(1).capitalize()
        year = match.group(2)
        return f"{month} {year}"
    return None


def collapse_period_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename data columns to their extracted period and sum columns that
    share the same period label.  Non-data columns are kept as-is.
    """
    id_cols = ["organisationunitid", "organisationunitname", "organisationunitcode"]
    rename_map: dict[str, str] = {}
    no_period: list[str] = []

    for col in df.columns:
        if col in id_cols:
            continue
        period = extract_period(col)
        if period:
            rename_map[col] = period
        else:
            no_period.append(col)

    if no_period:
        print(
            f"[warning] Could not extract a period from {len(no_period)} column(s); "
            f"they will be dropped:\n  " + "\n  ".join(no_period)
        )

    # Keep only id cols + mappable cols
    keep_cols = id_cols + list(rename_map.keys())
    df = df[keep_cols].rename(columns=rename_map)
    
    remaining_items = [item for item in list(df.columns) if item not in set(id_cols)]
    new_list = id_cols + remaining_items
    df = df[new_list]

    return df


def code_level(code) -> int | None:
    """Return hierarchy level based on code string length."""
    s = str(code).strip()
    if len(s) == 1:
        return 0  # Province
    if len(s) == 3:
        return 1  # District
    if len(s) == 5:
        return 2  # Municipality
    return None  # unknown


def split_by_level(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split DataFrame into province, district, municipality subsets."""
    levels = df["organisationunitcode"].astype(str).str.strip().str.len().map(
        {1: "province", 3: "district", 5: "municipality"}
    )
    provinces = df[levels == "province"].copy()
    districts = df[levels == "district"].copy()
    municipalities = df[levels == "municipality"].copy()
    return provinces, districts, municipalities


def build_province_lookup(provinces: pd.DataFrame) -> dict[str, tuple[str, str]]:
    """Map province_code_prefix -> (province_name, province_code)."""
    lookup = {}
    for _, row in provinces.iterrows():
        code = str(row["organisationunitcode"]).strip()
        lookup[code] = (row["organisationunitname"], code)
    return lookup


def build_district_lookup(districts: pd.DataFrame) -> dict[str, tuple[str, str]]:
    """Map district_code -> (district_name, district_code)."""
    lookup = {}
    for _, row in districts.iterrows():
        code = str(row["organisationunitcode"]).strip()
        lookup[code] = (row["organisationunitname"], code)
    return lookup


def enrich_districts(
    districts: pd.DataFrame,
    province_lookup: dict,
) -> pd.DataFrame:
    """Add province_name and province_code columns to district rows."""
    prov_names, prov_codes = [], []
    for code in districts["organisationunitcode"].astype(str).str.strip():
        # Province code is first digit of district code
        prov_key = code[0]
        if prov_key in province_lookup:
            name, pcode = province_lookup[prov_key]
        else:
            name, pcode = None, None
        prov_names.append(name)
        prov_codes.append(pcode)

    out = districts.copy()
    out.insert(2, "province_name", prov_names)
    out.insert(3, "province_code", prov_codes)
    return out


def enrich_municipalities(
    municipalities: pd.DataFrame,
    province_lookup: dict,
    district_lookup: dict,
) -> pd.DataFrame:
    """Add district_name, district_code, province_name, province_code to municipality rows."""
    dist_names, dist_codes = [], []
    prov_names, prov_codes = [], []

    for code in municipalities["organisationunitcode"].astype(str).str.strip():
        # District code = first 3 digits; Province code = first digit
        dist_key = code[:3]
        prov_key = code[0]

        if dist_key in district_lookup:
            dname, dcode = district_lookup[dist_key]
        else:
            dname, dcode = None, None

        if prov_key in province_lookup:
            pname, pcode = province_lookup[prov_key]
        else:
            pname, pcode = None, None

        dist_names.append(dname)
        dist_codes.append(dcode)
        prov_names.append(pname)
        prov_codes.append(pcode)

    out = municipalities.copy()
    out.insert(2, "district_name", dist_names)
    out.insert(3, "district_code", dist_codes)
    out.insert(4, "province_name", prov_names)
    out.insert(5, "province_code", prov_codes)
    return out

def convert_dates(df: pd.DataFrame) -> pd.DataFrame:
    """Rename columns in df based on Bikram Sambat dates to datetime conversion"""
    for period in df.columns:
        if period.split()[0] in NEPALI_MONTHS:
            m = NEPALI_MONTHS.index(period.split()[0]) + 1
            y = int(period.split()[1])
            ad_date = nepali_datetime.date(y, m, 12).to_datetime_date()
            df.rename(columns={period: ad_date}, inplace=True)

    return df

In [40]:
print("Poisonous: Splitting by administrative level...")
provinces_raw, districts_raw, municipalities_raw = split_by_level(df_pois[keep_cols_p])
print(f"  Provinces:     {len(provinces_raw):>6,} rows")
print(f"  Districts:     {len(districts_raw):>6,} rows")
print(f"  Municipalities:{len(municipalities_raw):>6,} rows")

unclassified = len(df_pois) - len(provinces_raw) - len(districts_raw) - len(municipalities_raw)
if unclassified:
    print(f"  [warning] {unclassified} rows could not be classified and were dropped.")

Poisonous: Splitting by administrative level...
  Provinces:          7 rows
  Districts:         77 rows
  Municipalities:   648 rows


In [41]:
# ---- Collapse period columns per level independently --------------------
print("Extracting and collapsing period columns...")
provinces_raw   = collapse_period_columns(provinces_raw)
districts_raw   = collapse_period_columns(districts_raw)
municipalities_raw = collapse_period_columns(municipalities_raw)

period_cols = [c for c in provinces_raw.columns if c not in ["organisationunitid", "organisationunitname", "organisationunitcode"]]
print(f"  {len(period_cols)} unique period(s) found: {', '.join(period_cols[:5])}" + "...")

Extracting and collapsing period columns...
  128 unique period(s) found: Mangsir 2081, Ashwin 2080, Asar 2078, Magh 2073, Kartik 2075...


In [42]:
# ---- Convert dates to AD --------------------
print("Converting dates...")
provinces   = convert_dates(provinces_raw)
districts   = convert_dates(districts_raw)
municipalities = convert_dates(municipalities_raw)

Converting dates...


In [43]:
province_lookup = build_province_lookup(provinces)
district_lookup = build_district_lookup(districts)

districts_enriched = enrich_districts(districts, province_lookup)
municipalities_enriched = enrich_municipalities(municipalities, province_lookup, district_lookup)

In [44]:
provinces_long = pd.melt(provinces,
        id_vars=["organisationunitid", "organisationunitname", "organisationunitcode"],
        var_name="month",
        value_name="p_cases")

provinces_long.head(3)

,organisationunitid,organisationunitname,organisationunitcode,month,p_cases
0,fvN7GZvNAOB,6 Karnali Province,6,2024-11-27,0.0
1,RVc3XoVoNRf,1 Koshi Province,1,2024-11-27,15.0
2,wtU6v09Kbe0,7 Sudurpashchim Province,7,2024-11-27,1.0


In [45]:
dist_long = pd.melt(districts_enriched,
        id_vars=['organisationunitid','organisationunitname','province_name',
              'province_code', 'organisationunitcode'],
        var_name="month",
        value_name="p_cases")

dist_long.head(3)

,organisationunitid,organisationunitname,province_name,province_code,organisationunitcode,month,p_cases
0,DLyaDAF00hm,204 MAHOTTARI,2 Madhesh Province,2,204,2024-11-27,2.0
1,vjL1uSC10Jh,701 BAJURA,7 Sudurpashchim Province,7,701,2024-11-27,0.0
2,CNFRaFUfLjo,111 JHAPA,1 Koshi Province,1,111,2024-11-27,0.0


In [46]:
mun_long = pd.melt(municipalities_enriched,
        id_vars=['organisationunitid', 'organisationunitname','district_name',
              'district_code','province_name','province_code','organisationunitcode'],
        var_name="month",
        value_name="p_cases")

mun_long.head(3)

,organisationunitid,organisationunitname,district_name,district_code,province_name,province_code,organisationunitcode,month,p_cases
0,YubZ3SsXpMk,10905 Falgunanda Rural Municipality,109 PANCHTHAR,109,1 Koshi Province,1,10905,2024-11-27,0.0
1,UYA23UmgomU,30802 Lalitpur Metropolitan City,308 LALITPUR,308,3 Bagmati Province,3,30802,2024-11-27,0.0
2,wXQf5ocsOhG,20804 Paterwa Sugauli Rural Municipality,208 PARSA,208,2 Madhesh Province,2,20804,2024-11-27,0.0


In [47]:
print("Non-Poisonous: Splitting by administrative level...")
provinces_raw, districts_raw, municipalities_raw = split_by_level(df_nonpois[keep_cols_np])
print(f"  Provinces:     {len(provinces_raw):>6,} rows")
print(f"  Districts:     {len(districts_raw):>6,} rows")
print(f"  Municipalities:{len(municipalities_raw):>6,} rows")

unclassified = len(df_nonpois) - len(provinces_raw) - len(districts_raw) - len(municipalities_raw)
if unclassified:
    print(f"  [warning] {unclassified} rows could not be classified and were dropped.")

Non-Poisonous: Splitting by administrative level...
  Provinces:          7 rows
  Districts:         77 rows
  Municipalities:   620 rows


In [48]:
# ---- Collapse period columns per level independently --------------------
print("Extracting and collapsing period columns...")
provinces_raw   = collapse_period_columns(provinces_raw)
districts_raw   = collapse_period_columns(districts_raw)
municipalities_raw = collapse_period_columns(municipalities_raw)

period_cols = [c for c in provinces_raw.columns if c not in ["organisationunitid", "organisationunitname", "organisationunitcode"]]
print(f"  {len(period_cols)} unique period(s) found: {', '.join(period_cols[:5])}" + "...")

Extracting and collapsing period columns...
  128 unique period(s) found: Bhadra 2077, Chaitra 2077, Kartik 2074, Magh 2079, Asar 2079...


In [49]:
# ---- Convert dates to AD --------------------
print("Converting dates...")
provinces   = convert_dates(provinces_raw)
districts   = convert_dates(districts_raw)
municipalities = convert_dates(municipalities_raw)

Converting dates...


In [50]:
np_provinces_long = pd.melt(provinces,
        id_vars=["organisationunitid", "organisationunitname", "organisationunitcode"],
        var_name="month",
        value_name="np_cases")

np_provinces_long.head(3)

,organisationunitid,organisationunitname,organisationunitcode,month,np_cases
0,fvN7GZvNAOB,6 Karnali Province,6,2020-08-28,52.0
1,RVc3XoVoNRf,1 Koshi Province,1,2020-08-28,228.0
2,wtU6v09Kbe0,7 Sudurpashchim Province,7,2020-08-28,19.0


In [51]:
np_dist_long = pd.melt(districts,
        id_vars=['organisationunitid','organisationunitname','organisationunitcode'],
        var_name="month",
        value_name="np_cases")

np_dist_long.head(3)

,organisationunitid,organisationunitname,organisationunitcode,month,np_cases
0,DLyaDAF00hm,204 MAHOTTARI,204,2020-08-28,0.0
1,vjL1uSC10Jh,701 BAJURA,701,2020-08-28,1.0
2,CNFRaFUfLjo,111 JHAPA,111,2020-08-28,128.0


In [52]:
np_mun_long = pd.melt(municipalities,
        id_vars=['organisationunitid','organisationunitname','organisationunitcode'],
        var_name="month",
        value_name="np_cases")

np_mun_long.head(3)

,organisationunitid,organisationunitname,organisationunitcode,month,np_cases
0,YubZ3SsXpMk,10905 Falgunanda Rural Municipality,10905,2020-08-28,4.0
1,UYA23UmgomU,30802 Lalitpur Metropolitan City,30802,2020-08-28,0.0
2,wXQf5ocsOhG,20804 Paterwa Sugauli Rural Municipality,20804,2020-08-28,0.0


In [53]:
prov_merged = provinces_long.merge(
    np_provinces_long,
    left_on=["organisationunitid", "organisationunitname", "organisationunitcode", "month"],
    right_on=["organisationunitid", "organisationunitname", "organisationunitcode", "month"],
    how='left'
)

dist_merged = dist_long.merge(
    np_dist_long,
    left_on=["organisationunitid", "organisationunitname", "organisationunitcode", "month"],
    right_on=["organisationunitid", "organisationunitname", "organisationunitcode", "month"],
    how='left'
)

mun_merged = mun_long.merge(
    np_mun_long,
    left_on=["organisationunitid", "organisationunitname", "organisationunitcode", "month"],
    right_on=["organisationunitid", "organisationunitname", "organisationunitcode", "month"],
    how='left'
)

In [54]:
proc_data_folder = "../proc_data"
out_province = "provinces.csv"
out_district = "districts.csv"
out_municipality = "municipalities.csv"

prov_merged.to_csv(os.path.join(proc_data_folder, out_province), index=False)
dist_merged.to_csv(os.path.join(proc_data_folder, out_district), index=False)
mun_merged.to_csv(os.path.join(proc_data_folder, out_municipality), index=False)

print(f"\nSaved:")
print(f"  {out_province}")
print(f"  {out_district}")
print(f"  {out_municipality}")
print("Done.")


Saved:
  provinces.csv
  districts.csv
  municipalities.csv
Done.


## Next, run the population.ipynb script to merge in the yearly population data